# LEGA-C DR3: galaxies lost for reasons other than being passive

- Passive and quality cuts as in `results/legac-dr3-2026-11-11/analysis.ipynb`.
- Photometry flags: COSMOS2015 clean cut against the fit rule of `scripts/cosmos_photometry.py`.

In [1]:
import json, warnings
from pathlib import Path
import astropy.units as u
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
from scipy.stats import ks_2samp
from specutils.utils.wcs_utils import air_to_vac

warnings.filterwarnings('ignore')
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/experiment.py').exists())
RAW, OUT = ROOT / 'data/raw', ROOT / 'results/dr3-more-galaxies-2026-11-16'
WINDOW, TELLURIC = (4074.5, 4666.1), (7590.0, 7660.0)
text = lambda v: (v.decode() if isinstance(v, bytes) else str(v)).strip()

cat = Table.read(RAW / 'legac_dr3/legac_DR3.fits', hdu=1).to_pandas()
for column in ('SPECT_ID', 'Filename'):
    cat[column] = cat[column].map(text)
phot = Table.read(RAW / 'cosmos2015/cosmos2015_legac_dr3_photometry_1arcsec.fits').to_pandas().set_index('LEGAC_INDEX')
P = cat.join(phot[['Area', 'Sat', 'Cfl', 'Deep', 'Flag', 'NUVMag', 'RMag', 'JMag']], how='inner')

def flag(path, column):
    t = Table.read(path)
    return dict(zip(np.char.strip(np.asarray(t['SPECT_ID']).astype(str)), np.asarray(t[column]).astype(int)))

P['warn25'] = P.SPECT_ID.map(flag(RAW / 'cosmos2025/cosmos2025_phot_legac_dr3_1arcsec.fits', 'warn-flag'))
P['classic'] = P.SPECT_ID.map(flag(RAW / 'cosmos2020/cosmos2020_classic_legac_dr3_1arcsec.fits', 'FlagCOMBINED'))
# Fit rule: COSMOS2025 where matched (warn-flag 0), otherwise Classic (FlagCOMBINED 0).
P['fittable'] = np.where(P.warn25.notna(), P.warn25 == 0, P.classic == 0)
P['photometry'] = np.where(P.warn25.notna(), 'cosmos2025', 'cosmos2020_classic')
P['c15_clean'] = (P.Area == 0) & (P.Sat == 0) & (P.Cfl == 1) & (P.Flag == 0)

- Quality, NUVrJ, [O II] EW and 3σ line cuts, unchanged.

In [2]:
quality = (P.FLAG_PRIMARY == 1) & (P.FLAG_SPEC == 0) & P.Z.notna() & P.SIGMA_STARS_PRIME.notna() & (P.FLAG_MORPH == 0)
quality &= (P.SN > 0) & (P.Z >= 0.6) & (P.Z < 1.0) & (P[['NUVMag', 'RMag', 'JMag']] > -40).all(axis=1)
P['nuv_r'], P['r_j'] = P.NUVMag - P.RMag, P.RMag - P.JMag
nuvrj = (P.nuv_r > 3 * P.r_j + 1) & (P.nuv_r > 3.1)
oii_ew = (P.OII_3727_EW < 5) | P.OII_3727_EW.isna()
detected = lambda line: (P[f'{line}_EW'] / P[f'{line}_EW_ERR']).abs() >= 3
no_line = ~(detected('OII_3727') | detected('OIII_5007'))
S = P[quality & nuvrj & oii_ew & no_line].copy()
chain = [('COSMOS2015 match', len(P)), ('quality', quality.sum()), ('NUVrJ quiescent', (quality & nuvrj).sum()),
         ('[O II] EW < 5 A', (quality & nuvrj & oii_ew).sum()), ('no 3 sigma [O II] or [O III]', len(S))]

- Good-pixel rest-frame span outside the telluric band; window coverage.

In [3]:
def rest_span(filename, z):
    with fits.open(RAW / 'legac_dr3/sp' / filename) as hdulist:
        wave, flux, err, qual = (np.asarray(hdulist[1].data[c][0], float) for c in ('WAVE', 'FLUX', 'ERR', 'QUAL'))
    good = (qual == 0) & (err > 0) & np.isfinite(flux) & ~((wave >= TELLURIC[0]) & (wave <= TELLURIC[1]))
    if not good.any():
        return np.nan, np.nan
    rest = air_to_vac(wave[good] * u.AA).to_value(u.AA) / (1 + z)
    return rest.min(), rest.max()

S[['rest_min', 'rest_max']] = [rest_span(f, z) for f, z in zip(S.Filename, S.Z)]
S['covers'] = (S.rest_min <= WINDOW[0]) & (S.rest_max >= WINDOW[1])
S['sigma_err_finite'] = np.isfinite(S.SIGMA_STARS_PRIME_ERR)
one_per_object = lambda d: d.sort_values(['SN', 'SPECT_ID'], ascending=[False, True]).drop_duplicates('OBJECT', keep='first')

- Reference chain: reproduces the 194.
- New chain: fittable photometry replaces the COSMOS2015 cut; one spectrum per OBJECT picked last.

In [4]:
old = one_per_object(S[S.c15_clean])
old = old[old.fittable & old.covers & old.sigma_err_finite]
reference = json.loads((ROOT / 'results/legac-dr3-2026-11-11/window-194/experiment.json').read_text())['targets']
assert sorted(old.SPECT_ID) == sorted(reference)

new = one_per_object(S[S.fittable & S.covers])
gained = new[~new.OBJECT.isin(old.OBJECT)].copy()
gained['change'] = np.select(
    [~gained.c15_clean & gained.sigma_err_finite, gained.c15_clean & ~gained.sigma_err_finite, ~gained.c15_clean],
    ['COSMOS2015 flag', 'NaN sigma-star error', 'COSMOS2015 flag + NaN sigma-star error'], 'spectrum picked after window cut')
chain += [('fittable photometry', S.fittable.sum()), ('covers window', (S.fittable & S.covers).sum()),
          ('one per OBJECT', len(new)), ('of which the reference 194', len(old)), ('gained', len(gained))]
chain += [(f'gained: {name}', n) for name, n in gained.change.value_counts().items()]
pd.DataFrame(chain, columns=['step', 'n']).to_csv(OUT / 'selection-counts.csv', index=False)
pd.DataFrame(chain, columns=['step', 'n'])

,step,n
0,COSMOS2015 match,4057
1,quality,2571
2,NUVrJ quiescent,1222
3,[O II] EW < 5 A,931
4,no 3 sigma [O II] or [O III],581
5,fittable photometry,523
6,covers window,371
7,one per OBJECT,348
8,of which the reference 194,194
9,gained,154


- COSMOS2015 flags of the 252 spectra the clean cut removes (VizieR J/ApJS/224/24: `Area` FLAG_HJMCC, `Sat` FLAG_PETER, `Cfl` FLAG_COSMOS, `Flag` FLAG_SHALLOW).

In [5]:
removed = S[~S.c15_clean]
flags = removed.groupby(['Area', 'Sat', 'Cfl', 'Flag']).agg(spectra=('fittable', 'size'), fittable=('fittable', 'sum')).reset_index()
flags.to_csv(OUT / 'cosmos2015-flags-removed.csv', index=False)
flags

,Area,Sat,Cfl,Flag,spectra,fittable
0,0,0,0,0,3,3
1,0,0,1,1,208,194
2,0,1,1,0,3,1
3,0,1,1,1,3,3
4,0,2,1,0,3,2
5,0,2,1,1,1,1
6,0,3,1,0,3,0
7,0,3,1,1,1,0
8,0,4,1,0,2,1
9,0,5,1,0,1,0


- σ★ error stand-in: median fractional error of the 194.

In [6]:
FRACTION = float((old.SIGMA_STARS_PRIME_ERR / old.SIGMA_STARS_PRIME).median())
FRACTION

0.03883352875709534

- Gained against the 194: 16, 50, 84 % and two-sample KS p.

In [7]:
for d in (old, gained):
    d['nuvrj_margin'] = np.minimum(d.nuv_r - (3 * d.r_j + 1), d.nuv_r - 3.1)
    d['oii_snr'] = (d.OII_3727_EW / d.OII_3727_EW_ERR).abs()
    d['oiii_snr'] = (d.OIII_5007_EW / d.OIII_5007_EW_ERR).abs()
COLUMNS = ['Z', 'SN', 'nuv_r', 'r_j', 'nuvrj_margin', 'OII_3727_EW', 'oii_snr', 'OIII_5007_EW', 'oiii_snr',
           'LICK_HD_A', 'SIGMA_STARS_PRIME']
rows = []
for column in COLUMNS:
    a, b = old[column].dropna(), gained[column].dropna()
    rows.append(dict(quantity=column, n_194=len(a), **dict(zip(['q16_194', 'q50_194', 'q84_194'], np.quantile(a, [.16, .5, .84]))),
                     n_gained=len(b), **dict(zip(['q16_gained', 'q50_gained', 'q84_gained'], np.quantile(b, [.16, .5, .84]))),
                     ks_p=ks_2samp(a, b).pvalue))
compare = pd.DataFrame(rows)
compare.to_csv(OUT / 'gained-vs-194.csv', index=False)
compare.round(3)

,quantity,n_194,q16_194,q50_194,q84_194,n_gained,q16_gained,q50_gained,q84_gained,ks_p
0,Z,194,0.669,0.728,0.842,154,0.665,0.728,0.836,0.128
1,SN,194,11.200,20.700,31.344,154,12.000,21.400,34.352,0.699
2,nuv_r,194,4.438,5.010,5.362,154,4.450,4.920,5.380,0.451
3,r_j,194,0.810,0.900,1.020,154,0.815,0.915,1.020,0.816
4,nuvrj_margin,194,0.638,1.280,1.721,154,0.610,1.185,1.706,0.282
5,OII_3727_EW,109,0.125,0.682,1.912,86,-0.069,0.331,1.052,0.025
6,oii_snr,109,0.374,1.336,2.502,86,0.353,0.870,2.161,0.037
7,OIII_5007_EW,117,0.070,0.338,0.728,92,0.089,0.273,0.618,0.235
8,oiii_snr,117,0.329,1.305,2.288,92,0.381,1.177,2.037,0.761
9,LICK_HD_A,169,-0.762,0.331,1.878,144,-0.832,0.319,1.594,0.961


- Target list for the fits.

In [8]:
KEEP = ['SPECT_ID', 'OBJECT', 'Z', 'SN', 'change', 'photometry', 'Area', 'Sat', 'Cfl', 'Deep', 'Flag', 'nuv_r', 'r_j',
        'OII_3727_EW', 'OII_3727_EW_ERR', 'OIII_5007_EW', 'OIII_5007_EW_ERR', 'SIGMA_STARS_PRIME', 'SIGMA_STARS_PRIME_ERR',
        'rest_min', 'rest_max']
gained = gained.sort_values('SPECT_ID')
gained[KEEP].to_csv(OUT / 'gained-per-galaxy.csv', index=False)
(OUT / 'targets.json').write_text(json.dumps({'targets': [
    {'spect_id': r.SPECT_ID, 'object_id': int(r.OBJECT), 'change': r.change} for r in gained.itertuples()]}, indent=1) + '\n')
gained.change.value_counts()

change
COSMOS2015 flag                           134
NaN sigma-star error                       14
COSMOS2015 flag + NaN sigma-star error      4
spectrum picked after window cut            2
Name: count, dtype: int64